In [9]:
import gym
import pygame
import numpy as np

class AgentEnv(gym.Env):
    def __init__(self):
        super(AgentEnv, self).__init__()
        
        pygame.init()
        self.width = 800
        self.height = 600
        self.screen = pygame.display.set_mode((self.width, self.height))
        self.clock = pygame.time.Clock()
        self.fps = 20    # frame per second
        self.grid_size = 40
        self.grid_width = self.width//self.grid_size ## no. of grids in row
        self.grid_height = self.height//self.grid_size  ## no. of grids in column
        
        self.action_space = gym.spaces.Discrete(4)
        ## self.observation_space = gym.spaces.Tuple((gym.spaces.Discrete(self.grid_width),gym.spaces.Discrete(self.grid_height)))
        self.observation_space = gym.spaces.Box(np.array([0,0]),np.array([self.grid_width-1,self.grid_height-1]))
        
        self.agent_image = pygame.image.load('Documents\TANK.png')
        self.agent_image = pygame.transform.scale(self.agent_image,(self.grid_size ,self.grid_size))
        self.player_image = pygame.transform.rotate(self.agent_image, 180)
        self.player_pos = [0, 0]
        self.target_pos = [self.grid_width - 1, self.grid_height - 1]
        self.target_image = pygame.Rect(self.target_pos[0]*self.grid_size,self.target_pos[1]*self.grid_size,self.grid_size,self.grid_size)
        
        self.obstacle1_pos = [self.grid_width - 7, self.grid_height - 9]
        self.obstacle1_image = pygame.Rect(self.obstacle1_pos[0]*self.grid_size,self.obstacle1_pos[1]*self.grid_size,2*self.grid_size,2*self.grid_size)

    def step(self, action):
        
        if action == 0:  # up
            self.player_pos[1] = max(0, self.player_pos[1] - 1)
            reward = -np.sqrt((self.player_pos[0]-self.target_pos[0])**2 + (self.player_pos[1]-self.target_pos[1])**2)
        elif action == 1:  # down
            self.player_pos[1] = min(self.grid_height - 1, self.player_pos[1] + 1)
            reward = -np.sqrt((self.player_pos[0]-self.target_pos[0])**2 + (self.player_pos[1]-self.target_pos[1])**2)
        elif action == 2:  # left
            self.player_pos[0] = max(0, self.player_pos[0] - 1)
            reward = -np.sqrt((self.player_pos[0]-self.target_pos[0])**2 + (self.player_pos[1]-self.target_pos[1])**2)
        elif action == 3:  # right
            self.player_pos[0] = min(self.grid_width - 1, self.player_pos[0] + 1)
            reward = -np.sqrt((self.player_pos[0]-self.target_pos[0])**2 + (self.player_pos[1]-self.target_pos[1])**2)
        
        if self.player_pos == self.target_pos:
            reward = 1000
            done = True
        elif self.player_pos == self.obstacle1_pos or self.player_pos == self.obstacle1_pos + [1,0] or self.player_pos == self.obstacle1_pos + [0,1] or self.player_pos == self.obstacle1_pos +[1,1]:
            reward = -200
            done = False
        else:
            reward = -1
            done = False
        return self.player_pos, reward, done, {}
    
    def reset(self):
        self.player_pos = [0, 0]
        return self.player_pos

    def render(self,mode = "human"):
        self.screen.fill((255, 255, 255))
        self.screen.blit(self.player_image, (self.player_pos[0] * self.grid_size, self.player_pos[1] * self.grid_size))
        pygame.draw.rect(self.screen,(255,0,0),self.target_image)
        pygame.draw.rect(self.screen,(200,150,50),self.obstacle1_image)
        pygame.display.update()
        self.clock.tick(self.fps)


In [10]:
from stable_baselines3 import PPO

env = AgentEnv()   # Create the environment
# model = PPO('MlpPolicy', env, verbose=1)  # Create the PPO agent
# model.learn(total_timesteps=10000)    # Train the agent for 10000 timesteps
# model.save('Documents\Trained Model_2D\Agent_Target_Obstacle_10000_timesteps')
model = PPO.load('Documents\Trained Model_2D\Agent_Target_Obstacle_10000_timesteps.zip', env = env)

                  
obs = env.reset()
score_tot = 0
num_episodes = 15
for i in range(num_episodes):
    score = 0
    while True:
        action, _states = model.predict(obs)
        obs, rewards, done,info = env.step(action)
        env.render()
        score += rewards
        if done == True:
            break
    print('Score: ' + str(score))
    obs = env.reset()
    score_tot += score
print('Avg Score = ' + str(score_tot/num_episodes))
env.close()


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Score: 936
Score: 902
Score: 931
Score: 538
Score: 927
Score: 849
Score: 915
Score: 745
Score: 948
Score: 934
Score: 906
Score: 921
Score: 913
Score: 928
Score: 926
Avg Score = 881.2666666666667


In [ ]:
from stable_baselines3 import DQN

env = AgentEnv()   # Create the environment
model = DQN('MlpPolicy', env, verbose=1)  # Create the PPO agent
model.learn(total_timesteps=10000,log_interval=4)    # Train the agent for 10000 timesteps
model.save('Documents\Trained Model_2D\Agent_Target_10000_timesteps')
# model = DQN.load('Documents\Trained Model_2D\Agent_Target_10000_timesteps_DQN.zip', env = env)


# clock = pygame.time.Clock()                   
obs = env.reset()
score_tot = 0
num_episodes = 5
for i in range(num_episodes):
    score = 0
    while True:
        action, _x = model.predict(obs, deterministic=True)
        obs, rewards, done,info = env.step(action)
        env.render()
        score += rewards
        if done == True:
            break
    print('Score: ' + str(score))
    obs = env.reset()
    score_tot += score
print('Avg Score = ' + str(score_tot/num_episodes))
env.close()
